In [1]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Demo").master("local[*]").getOrCreate()

# Linear Regression: Improving the Model with Categorical Variables

In [2]:
model_output = "/home/jovyan/work/outputs/models/lr-model/"


## Load Dataset
Let's load the clean Airbnb dataset in again 
We created it in the previous notebook, it should exists in `/home/jovyan/work/datasets/output/airbnb/clean_data`

In [3]:
file_path = f"/home/jovyan/work/datasets/output/airbnb/clean_data"
airbnb_df = spark.read.parquet(file_path)
train_df, test_df = airbnb_df.randomSplit([.8, .2], seed=42)

### One Hot Encoder
* Extract all the string variables
* Create a column with Index for each of them to serve as the output of the StringIndexer
* Create a column with OHE for each of them to serve as the output of the One-Hot Encoder
* You need to use [StringIndexer](https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.ml.feature.StringIndexer.html?highlight=stringindexer#pyspark.ml.feature.StringIndexer) in order to map a string column of labels to an ML column of label indices.
*  Then, apply the [OneHotEncoder](https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.ml.feature.OneHotEncoder.html?highlight=onehotencoder#pyspark.ml.feature.OneHotEncoder) to the output of the StringIndexer.

In [4]:
from pyspark.ml.feature import StringIndexer

df = spark.createDataFrame(
    [(0, "a"), (1, "b"), (2, "c"), (3, "a"), (4, "a"), (5, "c")],
    ["id", "category"])

print("original")
df.show()
indexer = StringIndexer(inputCol="category", outputCol="categoryIndex")
indexed = indexer.fit(df).transform(df)
print("indexed")
indexed.show()

original
+---+--------+
| id|category|
+---+--------+
|  0|       a|
|  1|       b|
|  2|       c|
|  3|       a|
|  4|       a|
|  5|       c|
+---+--------+

indexed
+---+--------+-------------+
| id|category|categoryIndex|
+---+--------+-------------+
|  0|       a|          0.0|
|  1|       b|          2.0|
|  2|       c|          1.0|
|  3|       a|          0.0|
|  4|       a|          0.0|
|  5|       c|          1.0|
+---+--------+-------------+



In [5]:
from pyspark.ml.feature import OneHotEncoder

df = spark.createDataFrame([
    (0.0, 1.0),
    (1.0, 0.0),
    (2.0, 1.0),
    (0.0, 2.0),
    (0.0, 1.0),
    (2.0, 0.0)
], ["categoryIndex1", "categoryIndex2"])

print("original")
df.show()

encoder = OneHotEncoder(inputCols=["categoryIndex1", "categoryIndex2"],
                        outputCols=["categoryVec1", "categoryVec2"])
model = encoder.fit(df)
encoded = model.transform(df)
encoded.show()

original
+--------------+--------------+
|categoryIndex1|categoryIndex2|
+--------------+--------------+
|           0.0|           1.0|
|           1.0|           0.0|
|           2.0|           1.0|
|           0.0|           2.0|
|           0.0|           1.0|
|           2.0|           0.0|
+--------------+--------------+

+--------------+--------------+-------------+-------------+
|categoryIndex1|categoryIndex2| categoryVec1| categoryVec2|
+--------------+--------------+-------------+-------------+
|           0.0|           1.0|(2,[0],[1.0])|(2,[1],[1.0])|
|           1.0|           0.0|(2,[1],[1.0])|(2,[0],[1.0])|
|           2.0|           1.0|    (2,[],[])|(2,[1],[1.0])|
|           0.0|           2.0|(2,[0],[1.0])|    (2,[],[])|
|           0.0|           1.0|(2,[0],[1.0])|(2,[1],[1.0])|
|           2.0|           0.0|    (2,[],[])|(2,[0],[1.0])|
+--------------+--------------+-------------+-------------+



In [6]:
from pyspark.ml.feature import OneHotEncoder, StringIndexer

categorical_cols = [field for (field, dataType) in train_df.dtypes if dataType == "string"]
index_output_cols = [x + "Index" for x in categorical_cols]
ohe_output_cols = [x + "OHE" for x in categorical_cols]

string_indexer = StringIndexer(inputCols=categorical_cols, outputCols=index_output_cols, handleInvalid="skip")
ohe_encoder = OneHotEncoder(inputCols=index_output_cols, outputCols=ohe_output_cols)

## Vector Assembler
Now you should combine our OHE categorical features with our numeric features.
 * Extract numeric columns (except price, since it is the target one)
 * Use Vector Assembler once again to create the features vector

In [7]:
from pyspark.ml.feature import VectorAssembler

numeric_cols = [field for (field, dataType) in train_df.dtypes if ((dataType == "double") & (field != "price"))]
assembler_inputs = ohe_output_cols + numeric_cols
vec_assembler = VectorAssembler(inputCols=assembler_inputs, outputCol="features")

## Linear Regression
* Build Linear Regression object with price as label

In [8]:
from pyspark.ml.regression import LinearRegression

lr = LinearRegression(labelCol="price", featuresCol="features")

## Pipeline
 A [Pipeline](https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.ml.Pipeline.html?highlight=pipeline#pyspark.ml.Pipeline) is a way of organizing all of the steps.

In [9]:
from pyspark.ml import Pipeline

#stages should be indexer, ohe, vectorizing and finally linear regression
stages = [string_indexer, ohe_encoder, vec_assembler, lr]
pipeline = Pipeline(stages=stages)

pipeline_model = pipeline.fit(train_df)

## Saving Models
Training a model may be costly, so we can save it in case our cluster goes down so we don't have to recompute our results.

In [10]:
pipeline_model.write().overwrite().save(model_output)

In [11]:
model_output

'/home/jovyan/work/outputs/models/lr-model/'

## Loading models
If all your transformers/estimators are set into a Pipeline, and stored like that, you can always load the generic `PipelineModel` back in. Otherwise, you may need to know which kind of model was stored (LinearRegression, LogisticRegression etc...)

In [12]:
from pyspark.ml import PipelineModel

saved_pipeline_model = PipelineModel.load(model_output)

## Apply the Model to Test Set

In [19]:
pred_df = saved_pipeline_model.transform(test_df)

pred_df.select("features", "price", "prediction").show()

+--------------------+-----+------------------+
|            features|price|        prediction|
+--------------------+-----+------------------+
|(307,[0,1,87,220,...|185.0|129.66910132826888|
|(307,[0,1,153,220...|100.0|106.09899124327967|
|(307,[0,1,61,220,...|230.0| 148.5554022035758|
|(307,[0,1,61,229,...|110.0| 66.52905958659721|
|(307,[0,1,15,220,...| 90.0|141.84203489568426|
|(307,[0,1,15,220,...|122.0|130.60423352875296|
|(307,[0,1,15,220,...|130.0|155.23456934213937|
|(307,[0,1,15,221,...| 58.0| 84.91784341974744|
|(307,[0,1,118,220...|109.0|140.43609599796764|
|(307,[0,1,131,220...|144.0|186.17081764270188|
|(307,[0,1,45,221,...|160.0| 77.85862856954554|
|(307,[0,1,95,224,...|232.0| 156.0335354880117|
|(307,[0,1,2,222,2...| 56.0| 75.28146394059331|
|(307,[0,1,2,220,2...|140.0|116.56292960415703|
|(307,[0,1,2,224,2...|200.0|132.76923697808706|
|(307,[0,1,2,224,2...|165.0|151.97910499886711|
|(307,[0,1,2,220,2...|175.0|133.49883200930344|
|(307,[0,1,2,221,2...| 80.0| 71.74511868

## Evaluate the Model

In [ ]:
display(pred_df.select("price", "prediction"))

In [20]:
from pyspark.ml.evaluation import RegressionEvaluator

regression_evaluator = RegressionEvaluator(predictionCol="prediction", labelCol="price", metricName="rmse")

rmse = regression_evaluator.evaluate(pred_df)
r2 = regression_evaluator.setMetricName("r2").evaluate(pred_df)
print(f"RMSE is {rmse}")
print(f"R2 is {r2}")

RMSE is 42.44272503215197
R2 is 0.45982698252128995
